# 03 · Testes estatísticos

Testes para verificar, com números, o que a exploração sugere. Para cada teste, o notebook informa a hipótese, o teste escolhido e por que ele se aplica.

Os dados são **observações de países e anos**, então não são independentes entre si. Os p-valores são indicativos e devem ser lidos com cuidado; o objetivo é quantificar padrões, não provar causas.

**Dependências:** `src/preparo.py`, `scipy`, `scikit-learn`, `tslearn`.

In [1]:
import sys
from pathlib import Path

# permite importar src/ a partir de notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.metrics import adjusted_rand_score
from tslearn.clustering import TimeSeriesKMeans

from src.dados import carregar_medium, carregar_outras_variantes, paises
from src.preparo import carregar_paises, matriz_series, padronizar

medium = carregar_medium()
paises_medium = paises(medium)
d2024 = paises_medium[paises_medium["Time"] == 2024].copy()

def p_formatado(p):
    return "< 0,001" if p < 0.001 else f"{p:.3f}".replace(".", ",")

def holm(p_valores):
    # Correção de Holm: controla a taxa de erro familiar em vários testes
    p = np.asarray(p_valores, dtype=float)
    m = len(p)
    ordem = np.argsort(p)
    ajustado = np.empty(m)
    maximo = 0.0
    for i, idx in enumerate(ordem):
        maximo = max(maximo, (m - i) * p[idx])
        ajustado[idx] = min(1.0, maximo)
    return ajustado

/Users/marcosabilio/.pyenv/versions/3.12.3/lib/python3.12/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


## Fundamentos dos testes

Cada teste responde a uma pergunta com uma **hipótese nula** (H₀) e um **p-valor**: a probabilidade de observar um resultado tão extremo quanto o obtido, se H₀ fosse verdadeira. Um p-valor baixo (em geral abaixo de 0,05) é evidência contra H₀. Ele não mede o tamanho do efeito e não prova causa.

Escolhemos testes **não paramétricos**, que trabalham com postos em vez de médias e desvios, porque a fecundidade e a mortalidade infantil têm distribuição assimétrica (notebook 01). Quando vários testes são feitos ao mesmo tempo, a chance de um falso positivo cresce; por isso aplicamos a correção de Holm.

Os testes respondem a perguntas da teoria da transição demográfica: se os indicadores se movem juntos, se as regiões estão em fases diferentes e se a queda da fecundidade é generalizada.

## 1. Correlação de Spearman entre indicadores (2024)

**Hipótese:** fecundidade, mortalidade infantil e idade mediana estão associadas à expectativa de vida.

**Teste:** correlação de Spearman (ρ). Usamos postos, não valores, porque a fecundidade e a mortalidade infantil são assimétricas (ver notebook 01). Os p-valores são corrigidos pelo método de Holm, pois são vários testes ao mesmo tempo.

Atenção à migração: a correlação de Spearman com a fecundidade (ρ ≈ −0,33) é moderada, bem maior que a de Pearson mostrada no notebook 01 (≈ −0,12). Com dados muito assimétricos, o Pearson subestima a associação: outra razão para usar Spearman aqui.

In [2]:
pares = [
    ("TFR", "LEx"),
    ("TFR", "IMR"),
    ("LEx", "IMR"),
    ("TFR", "MedianAgePop"),
    ("LEx", "MedianAgePop"),
    ("TFR", "NetMigrations"),
]

linhas = []
for a, b in pares:
    d = d2024[[a, b]].dropna()
    rho, p = stats.spearmanr(d[a], d[b])
    linhas.append({"par": f"{a} × {b}", "n": len(d), "rho": rho, "p (bruto)": p})

spearman = pd.DataFrame(linhas)
spearman["p (Holm)"] = [p_formatado(p) for p in holm(spearman["p (bruto)"])]
spearman["rho"] = spearman["rho"].round(3)
spearman.drop(columns=["p (bruto)"])

,par,n,rho,p (Holm)
0,TFR × LEx,237,-0.773,"< 0,001"
1,TFR × IMR,237,0.779,"< 0,001"
2,LEx × IMR,237,-0.899,"< 0,001"
3,TFR × MedianAgePop,237,-0.866,"< 0,001"
4,LEx × MedianAgePop,237,0.820,"< 0,001"
5,TFR × NetMigrations,237,-0.327,"< 0,001"


## 2. Fecundidade difere entre regiões? (Kruskal-Wallis)

**Hipótese nula:** a distribuição da fecundidade em 2024 é a mesma em todas as regiões.

**Teste:** Kruskal-Wallis, versão não paramétrica da ANOVA. Não supõe normalidade, o que é importante aqui: a fecundidade é assimétrica.

**Tamanho do efeito:** ε² (épsilon ao quadrado), que vai de 0 a 1. Interpretação aproximada: 0,01 pequeno, 0,06 médio, 0,14 grande.

In [3]:
REGIAO_MAPA = {}
geo = medium[medium["LocTypeName"] == "Geographic region"][["LocID", "Location"]].drop_duplicates()
sub = medium[medium["LocTypeName"] == "Subregion"][["LocID", "ParentID"]].drop_duplicates()
nome_geo = dict(zip(geo["LocID"], geo["Location"]))
sub_para_regiao = {int(l): nome_geo.get(int(p)) for l, p in zip(sub["LocID"], sub["ParentID"])}
sub_para_regiao[918] = "Northern America"
d2024["regiao"] = d2024["ParentID"].map(lambda p: sub_para_regiao.get(int(p), nome_geo.get(int(p))) if pd.notna(p) else None)

def kruskal_por_regiao(coluna):
    d = d2024[["regiao", coluna]].dropna()
    grupos = [g[coluna].values for _, g in d.groupby("regiao")]
    H, p = stats.kruskal(*grupos)
    k, n = len(grupos), len(d)
    epsilon2 = (H - k + 1) / (n - k)
    return {"indicador": coluna, "H": round(H, 2), "gl": k - 1, "n": n,
            "ε²": round(epsilon2, 3), "p": p_formatado(p)}

pd.DataFrame([kruskal_por_regiao(c) for c in ["TFR", "LEx", "IMR", "MedianAgePop"]])

,indicador,H,gl,n,ε²,p
0,TFR,123.03,5,237,0.511,"< 0,001"
1,LEx,113.54,5,237,0.470,"< 0,001"
2,IMR,129.40,5,237,0.539,"< 0,001"
3,MedianAgePop,141.38,5,237,0.590,"< 0,001"


Os quatro indicadores diferem entre regiões, com efeito grande para fecundidade, mortalidade infantil e expectativa de vida. Com o teste global, não sabemos *quais* regiões diferem. Para isso, comparações par a par com correção de Holm.

In [4]:
def comparacoes_par_a_par(coluna):
    d = d2024[["regiao", coluna]].dropna()
    regioes = sorted(d["regiao"].unique())
    linhas = []
    for i, a in enumerate(regioes):
        for b in regioes[i + 1:]:
            x = d.loc[d["regiao"] == a, coluna]
            y = d.loc[d["regiao"] == b, coluna]
            if len(x) < 3 or len(y) < 3:
                continue
            _, p = stats.mannwhitneyu(x, y, alternative="two-sided")
            linhas.append({"A": a, "B": b, "mediana A": x.median(), "mediana B": y.median(), "p bruto": p})
    tab = pd.DataFrame(linhas).sort_values("p bruto").reset_index(drop=True)
    tab["p (Holm)"] = holm(tab["p bruto"])
    tab["significativo"] = tab["p (Holm)"] < 0.05
    return tab.drop(columns=["p bruto"]).round(2)

comparacoes_par_a_par("TFR")

,A,B,mediana A,mediana B,p (Holm),significativo
0,Africa,Europe,3.80,1.45,0.00,True
1,Africa,Latin America and the Caribbean,3.80,1.62,0.00,True
2,Africa,Asia,3.80,1.95,0.00,True
3,Europe,Oceania,1.45,2.60,0.00,True
4,Asia,Europe,1.95,1.45,0.00,True
5,Latin America and the Caribbean,Oceania,1.62,2.60,0.00,True
6,Africa,Northern America,3.80,1.41,0.00,True
7,Africa,Oceania,3.80,2.60,0.00,True
8,Europe,Latin America and the Caribbean,1.45,1.62,0.00,True
9,Northern America,Oceania,1.41,2.60,0.01,True


## 3. A fecundidade caiu entre 1950 e 2024? (Wilcoxon pareado)

**Hipótese nula:** a mediana da diferença de fecundidade entre 2024 e 1950 é zero.

**Teste:** Wilcoxon de postos com sinais, pareado: os mesmos países nos dois anos. O pareamento importa, porque comparar as duas distribuições como independentes perderia a informação de que cada país é seu próprio ponto de comparação.

In [5]:
antes = paises_medium[paises_medium["Time"] == 1950][["LocID", "TFR"]]
depois = paises_medium[paises_medium["Time"] == 2024][["LocID", "TFR"]]
par = antes.merge(depois, on="LocID", suffixes=("_1950", "_2024")).dropna()
diferenca = par["TFR_2024"] - par["TFR_1950"]

W, p = stats.wilcoxon(par["TFR_2024"], par["TFR_1950"])
r_rb = 1 - (2 * W) / (len(par) * (len(par) + 1) / 2)  # correlação bisserial de postos
resultado_wilcoxon = pd.DataFrame([{
    "países pareados": len(par),
    "mediana 1950": round(par["TFR_1950"].median(), 2),
    "mediana 2024": round(par["TFR_2024"].median(), 2),
    "países com queda": int((diferenca < 0).sum()),
    "W": W,
    "p": p_formatado(p),
    "r (bisserial)": round(abs(r_rb), 3),
}])
resultado_wilcoxon

,países pareados,mediana 1950,mediana 2024,países com queda,W,p,r (bisserial)
0,237,5.81,1.93,235,5.0,"< 0,001",1.0


A queda é clara: a grande maioria dos países teve fecundidade menor em 2024. O tamanho do efeito (r bisserial) indica o quanto da diferença é explicado pela direção da mudança, e não só pela magnitude.

## 4. Tendência de cada país (Kendall, 1950–2023)

**Pergunta:** para quantos países a fecundidade tem tendência de queda estatisticamente detectável?

**Teste:** para cada país, τ de Kendall entre o ano e o TFR. É um teste de tendência não paramétrico.

**Limitação:** as observações de um mesmo país são autocorrelacionadas (o TFR de um ano depende do anterior), e o teste assume independência. O p-valor fica **otimista**. Por isso a leitura aqui é descritiva: a proporção de países com tendência de queda, e não um p-valor isolado.

In [6]:
def tendencia(grupo):
    g = grupo.dropna(subset=["TFR"])
    tau, p = stats.kendalltau(g["Time"], g["TFR"])
    return pd.Series({"tau": tau, "p": p})

historico = paises_medium[paises_medium["Time"].between(1950, 2023)]
tend = historico.groupby("LocID").apply(tendencia, include_groups=False).reset_index()
tend = tend.merge(d2024[["LocID", "Location", "regiao"]], on="LocID")

resumo = pd.DataFrame({
    "países": [len(tend)],
    "tendência de queda (τ<0, p<0,05)": [int(((tend["tau"] < 0) & (tend["p"] < 0.05)).sum())],
    "tendência de alta (τ>0, p<0,05)": [int(((tend["tau"] > 0) & (tend["p"] < 0.05)).sum())],
    "sem tendência clara (p ≥ 0,05)": [int((tend["p"] >= 0.05).sum())],
})
resumo

,países,"tendência de queda (τ<0, p<0,05)","tendência de alta (τ>0, p<0,05)","sem tendência clara (p ≥ 0,05)"
0,237,230,2,4


In [7]:
por_regiao = (
    tend.assign(queda=(tend["tau"] < 0) & (tend["p"] < 0.05))
    .groupby("regiao")
    .agg(países=("LocID", "size"), com_queda=("queda", "sum"))
    .assign(proporcao=lambda d: (d["com_queda"] / d["países"]).round(2))
    .sort_values("proporcao", ascending=False)
)
por_regiao

,países,com_queda,proporcao
regiao,,,
Latin America and the Caribbean,50,50,1.00
Northern America,5,5,1.00
Oceania,23,23,1.00
Asia,51,50,0.98
Europe,50,49,0.98
Africa,58,53,0.91


## 5. Intervalo de confiança da mediana (bootstrap)

A mediana da fecundidade entre países em 2024 é um número único. O bootstrap mostra a incerteza dessa estimativa, sem supor uma distribuição específica. Usamos o método BCa, que corrige o viés e a assimetria do bootstrap simples.

In [8]:
tfr = d2024["TFR"].dropna().values
resultado = stats.bootstrap((tfr,), np.median, n_resamples=9999, confidence_level=0.95,
                            method="BCa", random_state=42)
pd.DataFrame([{
    "mediana": round(np.median(tfr), 3),
    "IC 95% inferior": round(resultado.confidence_interval.low, 3),
    "IC 95% superior": round(resultado.confidence_interval.high, 3),
}])

,mediana,IC 95% inferior,IC 95% superior
0,1.931,1.731,2.109


## 6. Os cenários de projeção são diferentes? (Friedman)

Os cenários de projeção são aplicados aos mesmos países, então o teste de Friedman (medidas repetidas) compara os cenários país a país. Usamos a população mundial no fim do século.

Como os cenários não são independentes entre si (a *Median PI* é o centro da distribuição de incerteza), o resultado mostra que eles diferem, não qual é o correto.

In [9]:
outras = carregar_outras_variantes()
cenarios = ["High", "Low", "Constant fertility", "Zero migration", "Constant mortality", "Instant replacement"]
pop2100 = (
    outras[(outras["Time"] == 2100) & (outras["LocTypeName"] == "Country/Area") & outras["Variant"].isin(cenarios)]
    .pivot_table(index="LocID", columns="Variant", values="TPopulation1July")
    .dropna()
)
estatistica, p = stats.friedmanchisquare(*[pop2100[c] for c in cenarios])
pd.DataFrame([{"países": len(pop2100), "cenários": len(cenarios), "χ²": round(estatistica, 1), "p": p_formatado(p)}])

,países,cenários,χ²,p
0,237,6,703.2,"< 0,001"


## 7. Estabilidade da clusterização (DTW)

A clusterização do notebook 04 depende da semente inicial do k-means. Aqui medimos a estabilidade com 3 sementes, como um teste rápido; a estabilidade completa, com 10 sementes e consenso, está na seção 5 do notebook 04. Se os grupos mudam muito entre sementes, o resultado não é confiável.

**Teste:** índice de Rand ajustado (ARI) entre os agrupamentos gerados com sementes diferentes. ARI = 1 significa grupos idênticos; ARI perto de 0 significa grupos aleatórios.

**Cuidado:** o k-means com DTW é lento. Por isso usamos 3 sementes e um número limitado de iterações.

In [10]:
df_pad = padronizar(carregar_paises())
X, ids, nomes = matriz_series(df_pad)
K = 4

rotulos = {}
for semente in [0, 1, 2]:
    modelo = TimeSeriesKMeans(n_clusters=K, metric="dtw", random_state=semente, max_iter=20)
    rotulos[semente] = modelo.fit_predict(X)

pares_ari = [(a, b) for a in rotulos for b in rotulos if a < b]
pd.DataFrame([
    {"sementes": f"{a} × {b}", "ARI": round(adjusted_rand_score(rotulos[a], rotulos[b]), 3)}
    for a, b in pares_ari
])

,sementes,ARI
0,0 × 1,0.646
1,0 × 2,0.511
2,1 × 2,0.518


**Resultado:** o ARI fica entre 0,51 e 0,65. É uma estabilidade **moderada**: uma parte dos países troca de grupo conforme a semente. Isso não invalida o resultado, mas precisa ser dito na apresentação. Caminhos para melhorar: mais iterações (o teste usa `max_iter=20`), mais sementes e agregação dos resultados (consenso), ou uma escolha de K que seja estável entre sementes.

**Próximo teste sugerido:** repetir a comparação com o conjunto de 11 variáveis selecionadas pelo VIF (notebook 01), para ver se a estabilidade melhora.

## Resumo

| Pergunta | Teste | Resultado |
|---|---|---|
| Os indicadores se correlacionam? | Spearman, com correção de Holm | Correlações fortes entre TFR, LEx, IMR e idade mediana; migração moderada |
| A fecundidade difere entre regiões? | Kruskal-Wallis + Mann-Whitney com Holm | Diferença grande entre regiões |
| A fecundidade caiu desde 1950? | Wilcoxon pareado | Queda na grande maioria dos países |
| Quantos países têm tendência de queda? | Kendall por país | 230 de 237 países, com queda significativa (leitura descritiva) |
| Qual a incerteza da mediana? | Bootstrap BCa | Intervalo estreito para a mediana |
| Os cenários diferem? | Friedman | Diferença significativa |
| A clusterização é estável? | ARI entre 3 sementes | 0,51 a 0,65; com 10 sementes, ARI médio de 0,65 contra o consenso (notebook 04) |

Os números exatos são os das células acima. Se algum resultado mudar com outra semente, outro período ou outro indicador, a conclusão deve ser revista.